In [ ]:
# Cell 1: browser startup for filter debugging.
import importlib
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "browser").exists():
    ROOT = ROOT.parent.resolve()
if not (ROOT / "browser").exists():
    raise RuntimeError("Could not locate repo root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

browser_driver = importlib.import_module("browser.driver")
core_config = importlib.import_module("core.config")
filter_debug_tools = importlib.import_module("notebooks.filter_sync_debug.filter_debug_tools")

browser_driver = importlib.reload(browser_driver)
core_config = importlib.reload(core_config)
filter_debug_tools = importlib.reload(filter_debug_tools)

from browser.driver import build_driver
from core.config import load_app_config

app_config = load_app_config()
browser_cfg = dict(app_config["profile"]["browser"])
browser_cfg["headless"] = False
browser_cfg["start_maximized"] = True
old_user_data_dir = Path(r"D:\_Desktop\Projects\Automations prj\User Data")
if old_user_data_dir.exists():
    browser_cfg["user_data_dir"] = str(old_user_data_dir)

driver = build_driver(browser_cfg)

print(json.dumps({"root": str(ROOT), "browser": browser_cfg}, indent=2, ensure_ascii=False))


In [ ]:
# Cell 2: rerun this cell only after editing filter sync code.
import importlib
import json

browser_linkedin = importlib.import_module("browser.linkedin")
browser_linkedin_jobs = importlib.import_module("browser.linkedin_jobs")
filter_debug_tools = importlib.import_module("notebooks.filter_sync_debug.filter_debug_tools")
browser_linkedin = importlib.reload(browser_linkedin)
browser_linkedin_jobs = importlib.reload(browser_linkedin_jobs)
filter_debug_tools = importlib.reload(filter_debug_tools)

cell_input_json = r'''
{
  "filter_by": "Jobs",
  "filters": {
    "job_type": "Full-time"
  },
  "delay_seconds": 0.2,
  "verbose": true
}
'''

run_html_probe = True
job_type_html = '''
<li class="search-reusables__filter-value-item">
  <input name="job-type-filter-value" class="search-reusables__select-input" type="checkbox" value="F">
  <label>
    <p class="display-flex">
      <span class="t-14 t-black--light t-normal" aria-hidden="true">Full-time</span>
      <span class="visually-hidden">Filter by Full-time</span>
    </p>
  </label>
</li>
<li class="search-reusables__filter-value-item">
  <input name="job-type-filter-value" class="search-reusables__select-input" type="checkbox" value="I">
  <label>
    <p class="display-flex">
      <span class="t-14 t-black--light t-normal" aria-hidden="true">Internship</span>
      <span class="visually-hidden">Filter by Internship</span>
    </p>
  </label>
</li>
'''

cell_input = json.loads(cell_input_json)
if run_html_probe:
    print('=== HTML PROBE ===')
    print(json.dumps(filter_debug_tools.probe_job_type_html(job_type_html), indent=2, ensure_ascii=False))

result = filter_debug_tools.run_filter_sync_trace(
    driver,
    cell_input,
    delay_seconds=cell_input["delay_seconds"],
    verbose=cell_input["verbose"],
)

def _compact_rows(trace):
    rows = []
    for row in trace.get('rows', []):
        rows.append({
            'text': row.get('row_text', ''),
            'hint': row.get('hint_text', ''),
            'checked': row.get('checked', False),
            'decision': row.get('decision', ''),
            'reason': row.get('reason', ''),
        })
    return rows

print('=== JOB TYPE COMPACT BEFORE ===')
print(json.dumps(_compact_rows(result['job_type_before']), indent=2, ensure_ascii=False))
print('=== JOB TYPE COMPACT AFTER ===')
print(json.dumps(_compact_rows(result['job_type_after']), indent=2, ensure_ascii=False))
print('=== RESULT ===')
print(json.dumps(result["result"], indent=2, ensure_ascii=False))
